# Notebook 44 — The Character Spectrum

**Goal:** Label every Laplacian eigenvalue by its character quantum
numbers $(a_3, a_5, a_7)$, revealing *which faculty* each excitation
mode perturbs and *what it costs*.

For an abelian Cayley graph, each eigenvalue of the Laplacian corresponds
to a character $\chi$ of the group:

$$\lambda_\chi \;=\; \sum_{s \in S}\bigl(1 - \operatorname{Re}\,\chi(s)\bigr)$$

Since $\mathbb{Z}^*_{210} \cong \mathbb{Z}^*_3 \times \mathbb{Z}^*_5
\times \mathbb{Z}^*_7$, every character decomposes as
$\chi = \chi_3 \otimes \chi_5 \otimes \chi_7$ with quantum numbers
$(a_3, a_5, a_7)$ where $0 \le a_p < \varphi(p)$.

The 48 characters are the "Fourier modes" of the group. Each mode
identifies *which prime faculties are excited* and *by how much*.

## 1 — Setup

In [16]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import numpy as np
from fractions import Fraction
from collections import defaultdict
from itertools import product as cartesian_product
from solenoid_algebra import SA, PRIMES, P, PHI, PRIMORIALS

print(f"Z*_{{210}}: {PHI} elements")
print(f"Primes: {PRIMES}")
print(f"phi(p): {[SA.phi_p[p] for p in PRIMES]}")
print(f"Primitive roots: {SA.primitive_roots}")

Z*_{210}: 48 elements
Primes: [2, 3, 5, 7]
phi(p): [1, 2, 4, 6]
Primitive roots: {3: 2, 5: 2, 7: 3}


## 2 — The Character Labeling Formula

For an abelian group $G$ with generating set $S$ (closed under inversion),
the Cayley-graph Laplacian has eigenvalues indexed by the characters
$\hat{G}$:

$$\lambda_\chi = \sum_{s \in S}\bigl(1 - \operatorname{Re}\,\chi(s)\bigr).$$

This replaces anonymous matrix diagonalization with *labelled* modes.
Each character $\chi = \chi_3^{(a_3)} \otimes \chi_5^{(a_5)} \otimes
\chi_7^{(a_7)}$ carries three quantum numbers: $a_p = 0$ means the
faculty at prime $p$ is in its ground state; $a_p > 0$ means it is excited.

## 3 — Per-Prime Generators

Each per-prime generator acts on exactly one cyclic factor of the CRT
decomposition. The full generating set $S$ includes generators *and their
inverses* (unless self-inverse).

In [2]:
# Per-prime generators (same as NB43)
per_prime_gens = {}
for p in PRIMES:
    if p == 2:
        continue
    prim_root = SA.primitive_roots[p]
    residues = [1 if q != p else prim_root for q in PRIMES]
    gen = SA.reconstruct(residues)
    per_prime_gens[p] = gen

natural_gens = list(per_prime_gens.values())

# Full generating set S (with inverses)
gen_set_sep = []
for g in natural_gens:
    gen_set_sep.append(g)
    g_inv = SA.inverse(g)
    if g_inv != g:
        gen_set_sep.append(g_inv)

print("Per-prime generators:")
for p, g in per_prime_gens.items():
    g_inv = SA.inverse(g)
    self_inv = " (SELF-INVERSE)" if g == g_inv else f", inv = {g_inv}"
    print(f"  Prime {p}: gen = {g}{self_inv},  decomp = {SA.decompose(g)}")

print(f"\nFull generating set S: {gen_set_sep}")
print(f"|S| = {len(gen_set_sep)} (degree of Cayley graph)")

Per-prime generators:
  Prime 3: gen = 71 (SELF-INVERSE),  decomp = (1, 2, 1, 1)
  Prime 5: gen = 127, inv = 43,  decomp = (1, 1, 2, 1)
  Prime 7: gen = 31, inv = 61,  decomp = (1, 1, 1, 3)

Full generating set S: [71, 127, 43, 31, 61]
|S| = 5 (degree of Cayley graph)


## 4 — Per-Prime Spectral Decomposition

Each cyclic factor $\mathbb{Z}^*_p$ has its own Cayley-graph Laplacian
spectrum. For a cycle of length $\varphi(p)$ with generator $g$:

- **Degree 1** (self-inverse, $g = g^{-1}$):
  $\lambda_p(a) = 1 - \cos\!\bigl(2\pi a / \varphi(p)\bigr)$
- **Degree 2** (distinct inverse):
  $\lambda_p(a) = 2 - 2\cos\!\bigl(2\pi a / \varphi(p)\bigr)$

These are the building blocks of the full spectrum.

In [3]:
# Compute per-prime eigenvalue tables
per_prime_lambda = {}
for p_idx, p in enumerate(PRIMES):
    phi_p = SA.phi_p[p]
    per_prime_lambda[p] = {}
    if phi_p <= 1:
        per_prime_lambda[p][0] = 0.0
        continue
    gen = per_prime_gens[p]
    gen_inv = SA.inverse(gen)
    self_inv = (gen == gen_inv)
    for a in range(phi_p):
        idx = [0] * len(PRIMES)
        idx[p_idx] = a
        idx = tuple(idx)
        lam = 0.0
        chi_fwd = SA.character(idx, gen)
        lam += 1 - chi_fwd.real
        if not self_inv:
            chi_inv = SA.character(idx, gen_inv)
            lam += 1 - chi_inv.real
        per_prime_lambda[p][a] = lam

print("Per-prime Laplacian spectra:")
for p in PRIMES:
    phi_p = SA.phi_p[p]
    if phi_p <= 1:
        print(f"  Z*_{p}: phi = {phi_p}  (trivial, no spectrum)")
        continue
    gen = per_prime_gens[p]
    gen_inv = SA.inverse(gen)
    degree = 1 if gen == gen_inv else 2
    print(f"\n  Z*_{p}: phi = {phi_p}, degree = {degree}")
    for a in range(phi_p):
        v = per_prime_lambda[p][a]
        # Verify against cosine formula
        theta = 2 * np.pi * a / phi_p
        if degree == 1:
            expected = 1 - np.cos(theta)
        else:
            expected = 2 - 2 * np.cos(theta)
        match = abs(v - expected) < 1e-14
        print(f"    a = {a}:  lambda = {v:.6f}  "
              f"({'integer' if abs(v - round(v)) < 1e-14 else 'non-integer'})  "
              f"[formula check: {'OK' if match else 'FAIL'}]")

Per-prime Laplacian spectra:
  Z*_2: phi = 1  (trivial, no spectrum)

  Z*_3: phi = 2, degree = 1
    a = 0:  lambda = 0.000000  (integer)  [formula check: OK]
    a = 1:  lambda = 2.000000  (integer)  [formula check: OK]

  Z*_5: phi = 4, degree = 2
    a = 0:  lambda = 0.000000  (integer)  [formula check: OK]
    a = 1:  lambda = 2.000000  (integer)  [formula check: OK]
    a = 2:  lambda = 4.000000  (integer)  [formula check: OK]
    a = 3:  lambda = 2.000000  (integer)  [formula check: OK]

  Z*_7: phi = 6, degree = 2
    a = 0:  lambda = 0.000000  (integer)  [formula check: OK]
    a = 1:  lambda = 1.000000  (integer)  [formula check: OK]
    a = 2:  lambda = 3.000000  (integer)  [formula check: OK]
    a = 3:  lambda = 4.000000  (integer)  [formula check: OK]
    a = 4:  lambda = 3.000000  (integer)  [formula check: OK]
    a = 5:  lambda = 1.000000  (integer)  [formula check: OK]


## 5 — The Integer Spectrum

**Observation:** Every per-prime eigenvalue is an *exact integer*:

| Prime $p$ | $\varphi(p)$ | Spectrum |
|-----------|--------------|----------|
| 3 | 2 | $\{0, 2\}$ |
| 5 | 4 | $\{0, 2, 4, 2\}$ |
| 7 | 6 | $\{0, 1, 3, 4, 3, 1\}$ |

**Why?** Niven's theorem (1956): $\cos(r\pi) \in \mathbb{Q}$ only when
$r \in \{0, \pm\tfrac{1}{6}, \pm\tfrac{1}{4}, \pm\tfrac{1}{3},
\pm\tfrac{1}{2}, \pm\tfrac{2}{3}, \pm\tfrac{3}{4}, \pm\tfrac{5}{6},
\pm 1\}$.

Equivalently, all values of $\cos(2\pi k / n)$ are rational
if and only if $n \in \{1, 2, 3, 4, 6\}$.

For our primes: $\varphi(3) = 2$, $\varphi(5) = 4$, $\varphi(7) = 6$
— exactly within the Niven set $\{1,2,3,4,6\}$.

The Laplacian eigenvalue $2 - 2\cos(2\pi a / n)$ is then a
**rational combination of rational cosines**, hence an integer.

For the next prime $p = 11$: $\varphi(11) = 10$, and
$\cos(2\pi/10) = \cos 36° = (\sqrt{5}+1)/4 \notin \mathbb{Q}$.
**The integer spectrum fails.**

The first four primes $\{2, 3, 5, 7\}$ are the *maximal* prime set
whose Cayley-graph Laplacian has an all-integer spectrum.

In [4]:
# IDENTITY 46: Integer spectrum via Niven's theorem
print("IDENTITY 46: Integer Spectrum")
print("=" * 55)

niven_set = {1, 2, 3, 4, 6}
print(f"Niven set (n where all cos(2*pi*k/n) rational): {sorted(niven_set)}")
print()

all_integer = True
for p in PRIMES:
    phi_p = SA.phi_p[p]
    in_niven = phi_p in niven_set
    values = list(per_prime_lambda[p].values())
    is_int = all(abs(v - round(v)) < 1e-14 for v in values)
    all_integer = all_integer and is_int
    status = "INTEGER" if is_int else "NON-INTEGER"
    niven_status = "in Niven set" if in_niven else "NOT in Niven set"
    print(f"  Prime {p}: phi = {phi_p} ({niven_status}) -> spectrum {status}")
    if phi_p > 1:
        int_vals = [int(round(v)) for v in values]
        print(f"    Eigenvalues: {int_vals}")

# Check next prime
p_next = 11
phi_next = p_next - 1
cos_val = np.cos(2 * np.pi / phi_next)
is_rational = abs(cos_val - round(cos_val * 4) / 4) < 1e-10
print(f"\n  Next prime {p_next}: phi = {phi_next} "
      f"({'in' if phi_next in niven_set else 'NOT in'} Niven set)")
print(f"    cos(2*pi/{phi_next}) = {cos_val:.10f} "
      f"({'rational' if is_rational else 'IRRATIONAL'})")

print(f"\nAll separable eigenvalues integer: {all_integer}")
print(f"This is specific to primes {{2, 3, 5, 7}} — "
      f"fails for p = 11.")

IDENTITY 46: Integer Spectrum
Niven set (n where all cos(2*pi*k/n) rational): [1, 2, 3, 4, 6]

  Prime 2: phi = 1 (in Niven set) -> spectrum INTEGER
  Prime 3: phi = 2 (in Niven set) -> spectrum INTEGER
    Eigenvalues: [0, 2]
  Prime 5: phi = 4 (in Niven set) -> spectrum INTEGER
    Eigenvalues: [0, 2, 4, 2]
  Prime 7: phi = 6 (in Niven set) -> spectrum INTEGER
    Eigenvalues: [0, 1, 3, 4, 3, 1]

  Next prime 11: phi = 10 (NOT in Niven set)
    cos(2*pi/10) = 0.8090169944 (IRRATIONAL)

All separable eigenvalues integer: True
This is specific to primes {2, 3, 5, 7} — fails for p = 11.


## 6 — The Full 48-State Spectrum

The separable spectrum is the set of all sums
$\lambda = \lambda_3(a_3) + \lambda_5(a_5) + \lambda_7(a_7)$.

Since each summand is an integer (by Niven), the full spectrum is
$\subset \{0, 1, \ldots, 10\}$ where $10 = 2 + 4 + 4$ is the bandwidth.

In [5]:
# Compute all 48 character-labeled eigenvalues
all_indices = SA.all_character_indices()

char_eigenvalues = []
for idx in all_indices:
    lam = 0.0
    for s in gen_set_sep:
        chi_s = SA.character(idx, s)
        lam += 1 - chi_s.real
    char_eigenvalues.append((idx, lam))

char_eigenvalues.sort(key=lambda x: x[1])

# Verify separability: lambda = sum of per-prime contributions
max_sep_error = 0.0
for idx, lam in char_eigenvalues:
    a2, a3, a5, a7 = idx
    predicted = (per_prime_lambda[3][a3]
               + per_prime_lambda[5][a5]
               + per_prime_lambda[7][a7])
    max_sep_error = max(max_sep_error, abs(lam - predicted))

print("Full 48-state separable spectrum:")
print(f"  {'(a2,a3,a5,a7)':>16s}  {'lambda':>7s}  Excited faculties")
print(f"  {'-'*16}  {'-'*7}  {'-'*40}")

for idx, lam in char_eigenvalues:
    a2, a3, a5, a7 = idx
    excited = []
    if a3 != 0: excited.append(f"p3({a3})")
    if a5 != 0: excited.append(f"p5({a5})")
    if a7 != 0: excited.append(f"p7({a7})")
    label = " + ".join(excited) if excited else "GROUND STATE"
    print(f"  {str(idx):>16s}  {lam:>7.0f}  {label}")

print(f"\nSeparability: max error = {max_sep_error:.1e}  (EXACT)")
print(f"Spectrum range: [0, {max(lam for _, lam in char_eigenvalues):.0f}]")
print(f"All integer: {all(abs(lam - round(lam)) < 1e-14 for _, lam in char_eigenvalues)}")

Full 48-state separable spectrum:
     (a2,a3,a5,a7)   lambda  Excited faculties
  ----------------  -------  ----------------------------------------
      (0, 0, 0, 0)        0  GROUND STATE
      (0, 0, 0, 1)        1  p7(1)
      (0, 0, 0, 5)        1  p7(5)
      (0, 0, 3, 0)        2  p5(3)
      (0, 0, 1, 0)        2  p5(1)
      (0, 1, 0, 0)        2  p3(1)
      (0, 0, 0, 2)        3  p7(2)
      (0, 0, 3, 1)        3  p5(3) + p7(1)
      (0, 0, 1, 1)        3  p5(1) + p7(1)
      (0, 1, 0, 1)        3  p3(1) + p7(1)
      (0, 0, 0, 4)        3  p7(4)
      (0, 0, 1, 5)        3  p5(1) + p7(5)
      (0, 0, 3, 5)        3  p5(3) + p7(5)
      (0, 1, 0, 5)        3  p3(1) + p7(5)
      (0, 1, 3, 0)        4  p3(1) + p5(3)
      (0, 0, 0, 3)        4  p7(3)
      (0, 0, 2, 0)        4  p5(2)
      (0, 1, 1, 0)        4  p3(1) + p5(1)
      (0, 0, 3, 2)        5  p5(3) + p7(2)
      (0, 1, 3, 1)        5  p3(1) + p5(3) + p7(1)
      (0, 0, 1, 2)        5  p5(1) + p7(2)
      (0, 0

## 7 — Faculty Cost Hierarchy and the Spectral Gap

The spectral gap $\lambda_1$ is the cheapest nonzero excitation — the
minimum energy cost to perturb the system away from perfect alignment
(the zero mode).

**Which faculty is cheapest to excite?**

In [6]:
# IDENTITY 47: Faculty cost hierarchy
print("IDENTITY 47: Faculty Cost Hierarchy")
print("=" * 55)

# Spectral gap
nonzero = [(idx, lam) for idx, lam in char_eigenvalues if lam > 1e-10]
gap_idx, gap_val = nonzero[0]
a2, a3, a5, a7 = gap_idx

print(f"Spectral gap: lambda_1 = {gap_val:.0f}")
print(f"Gap character: {gap_idx}")
print(f"  a_3 = {a3}  (prime 3: {'ground' if a3 == 0 else 'EXCITED'})")
print(f"  a_5 = {a5}  (prime 5: {'ground' if a5 == 0 else 'EXCITED'})")
print(f"  a_7 = {a7}  (prime 7: {'ground' if a7 == 0 else 'EXCITED'})")

# Degeneracy at the gap
gap_modes = [(idx, lam) for idx, lam in nonzero if abs(lam - gap_val) < 1e-10]
print(f"  Degeneracy: {len(gap_modes)} "
      f"(characters: {[idx for idx, _ in gap_modes]})")

# Per-faculty minimum cost
print(f"\nMinimum excitation cost per faculty:")
faculty_costs = {}
for p_idx, p in enumerate(PRIMES):
    phi_p = SA.phi_p[p]
    if phi_p <= 1:
        print(f"  Prime {p}: trivial (no excited states)")
        continue
    costs = [(a, per_prime_lambda[p][a])
             for a in range(phi_p) if a != 0]
    min_a, min_cost = min(costs, key=lambda x: x[1])
    faculty_costs[p] = min_cost
    print(f"  Prime {p}: cost = {min_cost:.0f}  (a = {min_a})")

print(f"\nCost ordering: p7 ({faculty_costs[7]:.0f}) "
      f"< p3 ({faculty_costs[3]:.0f}) "
      f"= p5 ({faculty_costs[5]:.0f})")
print(f"The outermost orbit (prime 7) is cheapest to excite.")

IDENTITY 47: Faculty Cost Hierarchy
Spectral gap: lambda_1 = 1
Gap character: (0, 0, 0, 1)
  a_3 = 0  (prime 3: ground)
  a_5 = 0  (prime 5: ground)
  a_7 = 1  (prime 7: EXCITED)
  Degeneracy: 2 (characters: [(0, 0, 0, 1), (0, 0, 0, 5)])

Minimum excitation cost per faculty:
  Prime 2: trivial (no excited states)
  Prime 3: cost = 2  (a = 1)
  Prime 5: cost = 2  (a = 3)
  Prime 7: cost = 1  (a = 1)

Cost ordering: p7 (1) < p3 (2) = p5 (2)
The outermost orbit (prime 7) is cheapest to excite.


## 8 — Degeneracy Structure

Eleven distinct energy levels accommodate 48 states. The degeneracy at
energy $E$ equals the number of ways to partition $E$ into per-prime
contributions — a discrete convolution of per-prime multiplicity
functions.

In [7]:
# IDENTITY 48: Degeneracy convolution
print("IDENTITY 48: Palindromic Degeneracy")
print("=" * 55)

tol = 1e-8
eigenvalue_groups = defaultdict(list)
for idx, lam in char_eigenvalues:
    E = int(round(lam))
    eigenvalue_groups[E].append(idx)

bandwidth = max(eigenvalue_groups.keys())
deg_seq = [len(eigenvalue_groups.get(E, [])) for E in range(bandwidth + 1)]

print(f"Distinct energy levels: {len(eigenvalue_groups)} "
      f"(from {PHI} characters)")
print(f"Bandwidth: {bandwidth}\n")

print(f"  {'E':>3s}  {'d(E)':>5s}  {'d(B-E)':>6s}  "
      f"{'Palindromic':>11s}  Characters")
print(f"  {'-'*3}  {'-'*5}  {'-'*6}  {'-'*11}  {'-'*40}")

palindromic = True
for E in range(bandwidth + 1):
    d_E = deg_seq[E]
    d_mirror = deg_seq[bandwidth - E]
    match = d_E == d_mirror
    if not match:
        palindromic = False
    chars = eigenvalue_groups.get(E, [])
    if len(chars) <= 3:
        char_str = str(chars)
    else:
        char_str = f"{chars[:2]} ... (+{len(chars)-2})"
    print(f"  {E:>3d}  {d_E:>5d}  {d_mirror:>6d}  "
          f"{'OK' if match else 'FAIL':>11s}  {char_str}")

print(f"\nDegeneracy sequence: {deg_seq}")
print(f"Sum: {sum(deg_seq)} (should be {PHI})")
print(f"Palindromic: d(E) = d({bandwidth} - E): {palindromic}")
print(f"Peak degeneracy: {max(deg_seq)} at E = {deg_seq.index(max(deg_seq))} "
      f"(bandwidth midpoint)")

# Why palindromic: Laplacian symmetry
# For character chi, the conjugate chi_bar gives lambda_max - lambda_chi
print(f"\nOrigin: if chi gives lambda, then the 'complementary' character")
print(f"chi_bar gives {bandwidth} - lambda.  Since characters pair up,")
print(f"d(E) = d({bandwidth} - E).")

IDENTITY 48: Palindromic Degeneracy
Distinct energy levels: 11 (from 48 characters)
Bandwidth: 10

    E   d(E)  d(B-E)  Palindromic  Characters
  ---  -----  ------  -----------  ----------------------------------------
    0      1       1           OK  [(0, 0, 0, 0)]
    1      2       2           OK  [(0, 0, 0, 1), (0, 0, 0, 5)]
    2      3       3           OK  [(0, 0, 3, 0), (0, 0, 1, 0), (0, 1, 0, 0)]
    3      8       8           OK  [(0, 0, 0, 2), (0, 0, 3, 1)] ... (+6)
    4      4       4           OK  [(0, 1, 3, 0), (0, 0, 0, 3)] ... (+2)
    5     12      12           OK  [(0, 0, 3, 2), (0, 1, 3, 1)] ... (+10)
    6      4       4           OK  [(0, 0, 1, 3), (0, 0, 3, 3)] ... (+2)
    7      8       8           OK  [(0, 1, 3, 2), (0, 0, 2, 2)] ... (+6)
    8      3       3           OK  [(0, 0, 2, 3), (0, 1, 1, 3), (0, 1, 3, 3)]
    9      2       2           OK  [(0, 1, 2, 2), (0, 1, 2, 4)]
   10      1       1           OK  [(0, 1, 2, 3)]

Degeneracy sequence: [1, 2, 

## 9 — Coupled Generators: Breaking Separability

Under per-prime generators, the spectrum is a tensor sum:
$\lambda = \lambda_3(a_3) + \lambda_5(a_5) + \lambda_7(a_7)$.

What happens when generators act on *multiple primes simultaneously*?

Max-order generators ($\operatorname{ord} = 12 = \lambda(210)$) have
nontrivial components in every cyclic factor. The character formula
still applies — but the eigenvalues are no longer sums of independent
per-prime contributions. The faculties become **entangled**.

In [8]:
# Coupled generators (same as NB43)
max_gens = SA.max_order_generators
found = None
for i, g1 in enumerate(max_gens):
    for j, g2 in enumerate(max_gens[i+1:], i+1):
        for g3 in max_gens[j+1:]:
            if SA.generates_group([g1, g2, g3]):
                found = [g1, g2, g3]
                break
        if found:
            break
    if found:
        break

gen_set_coupled = []
for g in found:
    gen_set_coupled.append(g)
    g_inv = SA.inverse(g)
    if g_inv != g:
        gen_set_coupled.append(g_inv)

print(f"Coupled generators: {found}")
print(f"Decompositions: {[SA.decompose(g) for g in found]}")
print(f"Full S (with inverses): {gen_set_coupled}")
print(f"|S| = {len(gen_set_coupled)}")

# Character-labeled eigenvalues
char_eigenvalues_coupled = []
for idx in all_indices:
    lam = 0.0
    for s in gen_set_coupled:
        chi_s = SA.character(idx, s)
        lam += 1 - chi_s.real
    char_eigenvalues_coupled.append((idx, lam))

char_eigenvalues_coupled.sort(key=lambda x: x[1])

print(f"\nCoupled spectrum (first 20 of {len(char_eigenvalues_coupled)}):")
print(f"  {'(a2,a3,a5,a7)':>16s}  {'lambda':>10s}  Excited faculties")
print(f"  {'-'*16}  {'-'*10}  {'-'*40}")
for idx, lam in char_eigenvalues_coupled[:20]:
    a2, a3, a5, a7 = idx
    excited = []
    if a3 != 0: excited.append(f"p3({a3})")
    if a5 != 0: excited.append(f"p5({a5})")
    if a7 != 0: excited.append(f"p7({a7})")
    label = " + ".join(excited) if excited else "GROUND STATE"
    print(f"  {str(idx):>16s}  {lam:>10.6f}  {label}")

Coupled generators: [17, 23, 37]
Decompositions: [(1, 2, 2, 3), (1, 2, 3, 2), (1, 1, 2, 2)]
Full S (with inverses): [17, 173, 23, 137, 37, 193]
|S| = 6

Coupled spectrum (first 20 of 48):
     (a2,a3,a5,a7)      lambda  Excited faculties
  ----------------  ----------  ----------------------------------------
      (0, 0, 0, 0)    0.000000  GROUND STATE
      (0, 1, 1, 2)    0.803848  p3(1) + p5(1) + p7(2)
      (0, 1, 3, 4)    0.803848  p3(1) + p5(3) + p7(4)
      (0, 0, 2, 4)    3.000000  p5(2) + p7(4)
      (0, 0, 2, 2)    3.000000  p5(2) + p7(2)
      (0, 0, 0, 3)    4.000000  p7(3)
      (0, 1, 0, 3)    4.000000  p3(1) + p7(3)
      (0, 1, 2, 0)    4.000000  p3(1) + p5(2)
      (0, 0, 3, 1)    4.267949  p5(3) + p7(1)
      (0, 1, 3, 1)    4.267949  p3(1) + p5(3) + p7(1)
      (0, 0, 1, 5)    4.267949  p5(1) + p7(5)
      (0, 0, 3, 2)    4.267949  p5(3) + p7(2)
      (0, 0, 1, 4)    4.267949  p5(1) + p7(4)
      (0, 1, 1, 5)    4.267949  p3(1) + p5(1) + p7(5)
      (0, 0, 2, 5)    

## 10 — The Coupled Gap: $6 - 3\sqrt{3}$

The separable spectrum lives in $\mathbb{Z}$.
The coupled spectrum lives in $\mathbb{Z}[\sqrt{3}]$.

The irrationality $\sqrt{3}$ is the algebraic fingerprint of
inter-faculty entanglement. It arises because the coupled generators
produce phase contributions of $\pm\pi/6$ at character $(0,1,1,2)$,
and $\cos(\pi/6) = \sqrt{3}/2$.

Specifically, each of the three generator–inverse pairs contributes
$2(1 - \cos\pi/6) = 2 - \sqrt{3}$ to the eigenvalue. With three
such pairs:

$$\lambda_{\text{gap}} = 3(2 - \sqrt{3}) = 6 - 3\sqrt{3}.$$

In [9]:
# IDENTITY 49: Coupled gap = 6 - 3*sqrt(3)
print("IDENTITY 49: Algebraic Irrationality Under Coupling")
print("=" * 55)

# Coupled gap
nonzero_c = [(idx, lam) for idx, lam in char_eigenvalues_coupled
             if lam > 1e-10]
gap_c_idx, gap_c_val = nonzero_c[0]

# Exact value
exact_gap = 6 - 3 * np.sqrt(3)
error = abs(gap_c_val - exact_gap)

print(f"Coupled spectral gap: {gap_c_val:.10f}")
print(f"Predicted: 6 - 3*sqrt(3) = {exact_gap:.10f}")
print(f"Error: {error:.2e}")
print(f"Gap character: {gap_c_idx}")

# Show the phase structure
print(f"\nPhase structure of the gap mode:")
for s in gen_set_coupled:
    chi = SA.character(gap_c_idx, s)
    phase = np.angle(chi)
    contribution = 1 - chi.real
    print(f"  s = {s:>3d}  chi = {chi.real:+.6f}{chi.imag:+.6f}i  "
          f"phase = {phase/np.pi:.4f}*pi  "
          f"1-Re = {contribution:.6f}")
print(f"  Total = {gap_c_val:.6f}")

# All irrational eigenvalues in the coupled spectrum
print(f"\nCoupled eigenvalue families:")
sqrt3 = np.sqrt(3)
families = {
    "6 - 3*sqrt(3)": 6 - 3*sqrt3,
    "6 - sqrt(3)": 6 - sqrt3,
    "integer": None,
    "6 + sqrt(3)": 6 + sqrt3,
    "6 + 3*sqrt(3)": 6 + 3*sqrt3,
}

coupled_lookup = {idx: lam for idx, lam in char_eigenvalues_coupled}
for idx, lam in char_eigenvalues_coupled:
    for name, val in families.items():
        if val is not None and abs(lam - val) < 1e-10:
            break

# Classify each coupled eigenvalue
classified = defaultdict(list)
for idx, lam in char_eigenvalues_coupled:
    found_family = False
    for name, val in families.items():
        if val is not None and abs(lam - val) < 1e-10:
            classified[name].append((idx, lam))
            found_family = True
            break
    if not found_family:
        # Check if integer
        if abs(lam - round(lam)) < 1e-10:
            classified[f"integer ({int(round(lam))})"].append((idx, lam))
        else:
            classified[f"other ({lam:.6f})"].append((idx, lam))

for name in sorted(classified.keys(), key=lambda k: classified[k][0][1]):
    modes = classified[name]
    char_strs = [str(idx) for idx, _ in modes[:3]]
    more = f" ... (+{len(modes)-3})" if len(modes) > 3 else ""
    print(f"  {name:>20s}: {len(modes):>2d} modes  "
          f"lambda = {modes[0][1]:.6f}  "
          f"{', '.join(char_strs)}{more}")

# The contrast
print(f"\nSeparable spectrum: all in Z")
print(f"Coupled spectrum:   Z[sqrt(3)]")
print(f"sqrt(3) = cos(pi/6) fingerprint of six-fold (Z*_7) coupling")

IDENTITY 49: Algebraic Irrationality Under Coupling
Coupled spectral gap: 0.8038475773
Predicted: 6 - 3*sqrt(3) = 0.8038475773
Error: 9.99e-16
Gap character: (0, 1, 1, 2)

Phase structure of the gap mode:
  s =  17  chi = +0.866025+0.500000i  phase = 0.1667*pi  1-Re = 0.133975
  s = 173  chi = +0.866025-0.500000i  phase = -0.1667*pi  1-Re = 0.133975
  s =  23  chi = +0.866025-0.500000i  phase = -0.1667*pi  1-Re = 0.133975
  s = 137  chi = +0.866025+0.500000i  phase = 0.1667*pi  1-Re = 0.133975
  s =  37  chi = +0.866025-0.500000i  phase = -0.1667*pi  1-Re = 0.133975
  s = 193  chi = +0.866025+0.500000i  phase = 0.1667*pi  1-Re = 0.133975
  Total = 0.803848

Coupled eigenvalue families:
           integer (0):  1 modes  lambda = 0.000000  (0, 0, 0, 0)
         6 - 3*sqrt(3):  2 modes  lambda = 0.803848  (0, 1, 1, 2), (0, 1, 3, 4)
           integer (3):  2 modes  lambda = 3.000000  (0, 0, 2, 4), (0, 0, 2, 2)
           integer (4):  3 modes  lambda = 4.000000  (0, 0, 0, 3), (0, 1, 0, 3)

## 11 — Spectral Inversion Under Coupling

Coupling doesn't just shift eigenvalues — it *inverts the cost ordering*.

- The separable gap mode $(0,0,0,1)$ excites **one** faculty (prime 7)
  at cost $\lambda = 1$.
- The coupled gap mode $(0,1,1,2)$ excites **all three** faculties
  at cost $\lambda = 6 - 3\sqrt{3} \approx 0.804$.

The cheapest separable perturbation (single-faculty) becomes expensive
under coupling ($\lambda = 7$). The cheapest coupled perturbation
(all-faculty entangled) was expensive under separation ($\lambda = 7$).

The per-character shift $\delta_\chi = \lambda^{\text{coupled}}_\chi
- \lambda^{\text{sep}}_\chi$ measures how much coupling
*restructures* each mode's energy.

In [10]:
# IDENTITY 50: Spectral inversion
print("IDENTITY 50: Spectral Inversion Under Coupling")
print("=" * 55)

sep_lookup = {idx: lam for idx, lam in char_eigenvalues}

# The two gap modes
print("Gap mode comparison:")
sep_gap = (0, 0, 0, 1)
coup_gap = gap_c_idx
print(f"  Separable gap: chi = {sep_gap}, lambda_sep = {sep_lookup[sep_gap]:.0f}, "
      f"lambda_coup = {coupled_lookup[sep_gap]:.6f}")
print(f"  Coupled gap:   chi = {coup_gap}, lambda_sep = {sep_lookup[coup_gap]:.0f}, "
      f"lambda_coup = {coupled_lookup[coup_gap]:.6f}")
print(f"\n  Inversion: separable cheapest mode costs "
      f"{coupled_lookup[sep_gap]:.0f} under coupling")
print(f"             coupled cheapest mode costs "
      f"{sep_lookup[coup_gap]:.0f} under separation")

# Number of excited faculties in each gap mode
def count_excited(idx):
    return sum(1 for j, p in enumerate(PRIMES)
               if idx[j] != 0 and SA.phi_p[p] > 1)

n_sep = count_excited(sep_gap)
n_coup = count_excited(coup_gap)
print(f"\n  Separable gap excites {n_sep} faculty (isolated perturbation)")
print(f"  Coupled gap excites {n_coup} faculties (entangled perturbation)")

# Full shift table
print(f"\nPer-character shift delta = lambda_coupled - lambda_sep:")
print(f"  {'Character':>16s}  {'Sep':>5s}  {'Coupled':>10s}  "
      f"{'delta':>10s}  Faculties")
print(f"  {'-'*16}  {'-'*5}  {'-'*10}  {'-'*10}  {'-'*12}")

shifts = []
for idx in sorted(all_indices, key=lambda i: sep_lookup[i]):
    lam_s = sep_lookup[idx]
    lam_c = coupled_lookup[idx]
    delta = lam_c - lam_s
    n_exc = count_excited(idx)
    shifts.append((idx, lam_s, lam_c, delta, n_exc))

for idx, lam_s, lam_c, delta, n_exc in shifts:
    print(f"  {str(idx):>16s}  {lam_s:>5.0f}  {lam_c:>10.6f}  "
          f"{delta:>+10.6f}  {n_exc}")

# Summary statistics
print(f"\nShift statistics:")
deltas = [d for _, _, _, d, _ in shifts]
print(f"  Mean shift: {np.mean(deltas):+.6f}")
print(f"  Max positive: {max(deltas):+.6f}")
print(f"  Max negative: {min(deltas):+.6f}")
print(f"  Sum of shifts: {sum(deltas):.6f}")
print(f"  (Sum = |S_coupled| - |S_sep| = "
      f"{len(gen_set_coupled)} - {len(gen_set_sep)} = "
      f"{len(gen_set_coupled) - len(gen_set_sep)}, "
      f"times {PHI} should be "
      f"{(len(gen_set_coupled) - len(gen_set_sep)) * PHI})")

IDENTITY 50: Spectral Inversion Under Coupling
Gap mode comparison:
  Separable gap: chi = (0, 0, 0, 1), lambda_sep = 1, lambda_coup = 7.000000
  Coupled gap:   chi = (0, 1, 1, 2), lambda_sep = 7, lambda_coup = 0.803848

  Inversion: separable cheapest mode costs 7 under coupling
             coupled cheapest mode costs 7 under separation

  Separable gap excites 1 faculty (isolated perturbation)
  Coupled gap excites 3 faculties (entangled perturbation)

Per-character shift delta = lambda_coupled - lambda_sep:
         Character    Sep     Coupled       delta  Faculties
  ----------------  -----  ----------  ----------  ------------
      (0, 0, 0, 0)      0    0.000000   +0.000000  0
      (0, 0, 0, 1)      1    7.000000   +6.000000  1
      (0, 0, 0, 5)      1    7.000000   +6.000000  1
      (0, 0, 3, 0)      2    6.000000   +4.000000  1
      (0, 0, 1, 0)      2    6.000000   +4.000000  1
      (0, 1, 0, 0)      2    8.000000   +6.000000  1
      (0, 0, 0, 2)      3    9.000000   

## 12 — Cross-Verification

The character formula must reproduce the matrix Laplacian spectrum
exactly. This verifies both the character computation and the
matrix diagonalization from NB43.

In [11]:
print("Cross-verification: character formula vs matrix eigenvalues")
print("=" * 55)

def build_cayley_laplacian(gens):
    n = PHI
    A = np.zeros((n, n))
    for g in gens:
        g_inv = SA.inverse(g)
        for k in SA.Z_star:
            i, j = SA._idx[k], SA._idx[SA.multiply(k, g)]
            A[i, j] = 1
            A[j, i] = 1
            if g_inv != g:
                i2, j2 = SA._idx[k], SA._idx[SA.multiply(k, g_inv)]
                A[i2, j2] = 1
                A[j2, i2] = 1
    D = np.diag(A.sum(axis=1))
    return D - A, A

L_sep_mat, _ = build_cayley_laplacian(natural_gens)
L_coup_mat, _ = build_cayley_laplacian(found)

evals_sep_mat = np.sort(np.linalg.eigvalsh(L_sep_mat))
evals_coup_mat = np.sort(np.linalg.eigvalsh(L_coup_mat))

evals_sep_char = np.sort([lam for _, lam in char_eigenvalues])
evals_coup_char = np.sort([lam for _, lam in char_eigenvalues_coupled])

sep_diff = np.max(np.abs(evals_sep_mat - evals_sep_char))
coup_diff = np.max(np.abs(evals_coup_mat - evals_coup_char))

print(f"Separable: max |matrix - character| = {sep_diff:.2e}  "
      f"{'EXACT' if sep_diff < 1e-10 else 'MISMATCH'}")
print(f"Coupled:   max |matrix - character| = {coup_diff:.2e}  "
      f"{'EXACT' if coup_diff < 1e-10 else 'MISMATCH'}")

# Also verify the character table orthogonality
ct = SA.character_table()
gram = ct @ ct.conj().T / PHI
off_diag = np.abs(gram - np.eye(PHI)).max()
print(f"\nCharacter table orthogonality: {off_diag:.2e}  "
      f"{'PERFECT' if off_diag < 1e-10 else 'FAIL'}")

Cross-verification: character formula vs matrix eigenvalues
Separable: max |matrix - character| = 7.99e-15  EXACT
Coupled:   max |matrix - character| = 7.11e-15  EXACT

Character table orthogonality: 9.27e-16  PERFECT


## 13 — Scorecard

| # | Identity | Source | Free parameters |
|---|----------|--------|-----------------|
| **46** | All separable eigenvalues $\in \mathbb{Z}$; Niven's theorem: $\varphi(p) \in \{2,4,6\} \subset \{1,2,3,4,6\}$ | Number theory | 0 |
| **47** | Spectral gap $\lambda_1 = 1$ from pure $p = 7$ excitation; cost hierarchy $p_7 < p_3 = p_5$ | Character formula | 0 |
| **48** | Palindromic degeneracy $d(E) = d(10-E)$; 11 levels, peak 12 at $E = 5$ | Laplacian symmetry | 0 |
| **49** | Coupled gap $= 6 - 3\sqrt{3}$; coupling maps $\mathbb{Z} \to \mathbb{Z}[\sqrt{3}]$ | Phase interference | 0 |
| **50** | Spectral inversion: cheapest coupled mode $(0,1,1,2)$ excites all 3 faculties | Entangled characters | 0 |

**Running total: 50 structural identities, 0 free parameters, 2 honest nulls.**

In [14]:
checks = {}

# Identity 46: Integer spectrum (Niven)
checks["46: Integer spectrum (Niven)"] = all_integer

# Identity 47: Gap = 1, pure p7, cost hierarchy
gap_correct = abs(gap_val - 1.0) < 1e-14
gap_is_pure_p7 = (gap_idx[1] == 0 and gap_idx[2] == 0 and gap_idx[3] != 0)
cost_order = (faculty_costs[7] < faculty_costs[3]
              and abs(faculty_costs[3] - faculty_costs[5]) < 1e-14)
checks["47: Gap = 1, pure p7, cost order"] = (
    gap_correct and gap_is_pure_p7 and cost_order
)

# Identity 48: Palindromic degeneracy
checks["48: Palindromic degeneracy"] = palindromic

# Identity 49: Coupled gap = 6 - 3*sqrt(3)
checks["49: Coupled gap = 6 - 3*sqrt(3)"] = abs(gap_c_val - (6 - 3*np.sqrt(3))) < 1e-14

# Identity 50: Spectral inversion
count_excited = sum(1 for a in gap_c_idx if a != 0)
checks["50: Spectral inversion"] = (
    count_excited == 3
    and coupled_lookup.get(gap_c_idx) is not None
    and sep_lookup.get(gap_c_idx) is not None
)

print("=" * 60)
print("SCORECARD: The Character Spectrum")
print("=" * 60)
n_pass = 0
for name, ok in checks.items():
    tag = "PASS" if ok else "FAIL"
    if ok:
        n_pass += 1
    print(f"  [{tag}] Identity {name}")
print("-" * 60)
print(f"  {n_pass}/{len(checks)} identities verified.")

SCORECARD: The Character Spectrum
  [PASS] Identity 46: Integer spectrum (Niven)
  [PASS] Identity 47: Gap = 1, pure p7, cost order
  [PASS] Identity 48: Palindromic degeneracy
  [PASS] Identity 49: Coupled gap = 6 - 3*sqrt(3)
  [PASS] Identity 50: Spectral inversion
------------------------------------------------------------
  5/5 identities verified.


## Summary and Frontier

**What this notebook established:**

1. Every Laplacian eigenvalue carries a **character label** $(a_3, a_5, a_7)$
   that identifies which faculties are excited and by how much. The 48
   characters of $\mathbb{Z}^*_{210}$ are the quantum numbers of the system.

2. The separable spectrum is **all-integer**: $\{0, 1, \ldots, 10\}$.
   This is a number-theoretic consequence of Niven's theorem — it holds
   because $\varphi(p) \in \{2, 4, 6\}$ for $p \in \{3, 5, 7\}$,
   and these are exactly the values where $\cos(2\pi/n) \in \mathbb{Q}$.
   This property **fails for any larger prime set** (e.g., adding $p = 11$).

3. The spectral gap $\lambda_1 = 1$ comes from exciting **only prime 7**
   (the outermost orbit). The cost hierarchy is $p_7 = 1 < p_3 = p_5 = 2$:
   the physical level is cheapest to perturb.

4. Coupling maps the spectrum from $\mathbb{Z}$ to $\mathbb{Z}[\sqrt{3}]$.
   The coupled gap $6 - 3\sqrt{3}$ is an algebraic irrational — the
   fingerprint of inter-faculty phase interference.

5. Coupling **inverts** the cost ordering: the cheapest coupled mode
   $(0,1,1,2)$ excites all three faculties simultaneously, while the
   cheapest separable mode $(0,0,0,1)$ becomes expensive. Entangled
   perturbation costs *less* than isolated perturbation.

**Next frontier:**
- Solenoid metric from the Laplacian: is the character spectrum the
  spectral data of a Riemannian manifold?
- Zeta function: $\zeta_L(s) = \sum_\chi \lambda_\chi^{-s}$ —
  what does it encode?
- Heat kernel: $K(t) = \sum_\chi e^{-\lambda_\chi t}$ — diffusion
  dynamics on the state space.